# Seguridad y Logging en la API

En el notebook de cierre viste que el "Nivel 6: Produccion" incluye cosas como seguridad y monitoreo, y que no las cubrimos en el laboratorio principal. Este notebook te da una primera prueba real de dos de esas piezas, sin salir de FastAPI:

1. **Logging estructurado**: que la API deje un registro de cada prediccion que hace, con fecha y datos. Esto es exactamente lo que despues verias con `docker logs`.
2. **Variables de entorno**: sacar un dato sensible (una API key) del codigo y ponerlo afuera, donde se puede cambiar sin tocar el codigo fuente.
3. **Proteger un endpoint con una API key**: que `/predict` solo responda si quien pregunta manda la clave correcta.

Los tres temas se conectan: la clave sensible va a vivir en una variable de entorno, no hardcodeada; y cada intento de prediccion (exitoso o no) va a quedar registrado en el log.

## Preparacion rapida

Retomamos el modelo y la version basica de `main.py`, igual que en la Sesion 1.

In [6]:
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 500

ingresos = np.random.uniform(300, 5000, n)
edad = np.random.randint(18, 70, n)
historial_deuda = np.random.randint(0, 2, n)

score = (ingresos * 0.0006) - (historial_deuda * 1.5) + (edad * 0.01) + np.random.normal(0, 0.5, n)
aprobado = (score > score.mean()).astype(int)

df = pd.DataFrame({
    "ingresos": ingresos,
    "edad": edad,
    "historial_deuda": historial_deuda,
    "aprobado": aprobado
})

X = df[["ingresos", "edad", "historial_deuda"]]
y = df["aprobado"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LogisticRegression()
modelo.fit(X_train, y_train)
joblib.dump(modelo, "model.pkl")

print("Modelo listo. Precision en test:", modelo.score(X_test, y_test))

Modelo listo. Precision en test: 0.91


In [7]:
!pip install -q fastapi uvicorn pydantic

In [8]:
%%writefile main.py
import joblib
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="API de Scoring Crediticio")

modelo = joblib.load("model.pkl")


class ClienteInput(BaseModel):
    ingresos: float
    edad: int
    historial_deuda: int


@app.get("/health")
def health():
    return {"status": "online"}


@app.post("/predict")
def predict(cliente: ClienteInput):
    datos = [[cliente.ingresos, cliente.edad, cliente.historial_deuda]]
    prediccion = modelo.predict(datos)[0]
    decision = "Aprobado" if prediccion == 1 else "Rechazado"
    return {"edad": cliente.edad, "decision": decision}

Overwriting main.py


## 1: Logging estructurado

Hasta ahora, si algo pasaba dentro de la API, no quedaba ningun registro. En un servidor real esto es un problema: si un cliente se queja de que su prediccion parece rara, no tenes forma de revisar que datos mando ni que le respondiste.

Python trae el modulo `logging` en su libreria estandar, pensado exactamente para esto. Es preferible a usar `print()` porque permite:

- Agregar automaticamente fecha y hora a cada linea.
- Definir niveles (`INFO`, `WARNING`, `ERROR`) para filtrar que se muestra.
- Cambiar donde va el registro (consola, un archivo, un servicio externo) sin tocar el resto del codigo.

Un detalle importante para mas adelante: cuando esta API corra dentro de un contenedor Docker, todo lo que se imprima (ya sea con `print` o con `logging`) va a la salida estandar del contenedor, y se puede ver desde afuera con `docker logs`. Por eso el logging que armamos aca no es solo para Colab: es literalmente lo que verias en produccion.

Vamos a agregar un logger que registre cada prediccion, y un contador simple de cuantas predicciones se hicieron en total.

In [9]:
%%writefile main.py
import logging

import joblib
from fastapi import FastAPI
from pydantic import BaseModel



# Configuración del sistema de logging
logger = logging.getLogger("scoring_api")           # Creamos un logger específico para nuestra API
logger.setLevel(logging.INFO)                       # Indicamos que queremos registrar mensajes INFO o más importantes
logger.propagate = False                            # Evitamos que el mensaje se duplique en el logger principal

if not logger.handlers:                             # Evitamos crear varios handlers si el módulo se recarga
    handler = logging.StreamHandler()               # Envía los logs a la consola
    handler.setLevel(logging.INFO)                  # El handler muestra mensajes INFO o más importantes

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"  # Formato: fecha/hora | nivel | mensaje
    )
    handler.setFormatter(formatter)                 # Aplicamos el formato al handler
    logger.addHandler(handler)                      # Conectamos el handler con nuestro logger




app = FastAPI(title="API de Scoring Crediticio")
modelo = joblib.load("model.pkl")

total_predicciones = 0


class ClienteInput(BaseModel):
    ingresos: float
    edad: int
    historial_deuda: int


@app.get("/health")
def health():
    return {"status": "online"}


@app.post("/predict")
def predict(cliente: ClienteInput):
    global total_predicciones

    datos = [[cliente.ingresos, cliente.edad, cliente.historial_deuda]]
    prediccion = modelo.predict(datos)[0]
    decision = "Aprobado" if prediccion == 1 else "Rechazado"

    total_predicciones += 1
    logger.info(
        "Prediccion #%s | ingresos=%s edad=%s historial_deuda=%s -> %s",
        total_predicciones, cliente.ingresos, cliente.edad, cliente.historial_deuda, decision
    )

    return {"edad": cliente.edad, "decision": decision}

Overwriting main.py


Probamos la API y prestamos atencion a la salida de la celda: ademas del resultado de la prediccion, deberias ver la linea de log con fecha, hora y los datos del cliente.

In [10]:
import importlib
import main
importlib.reload(main)

from fastapi.testclient import TestClient
client = TestClient(main.app)

cliente_ejemplo = {"ingresos": 2500.0, "edad": 35, "historial_deuda": 0}
respuesta = client.post("/predict", json=cliente_ejemplo)
print("Respuesta:", respuesta.json())

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
2026-09-30 15:21:19,318 | INFO | Prediccion #1 | ingresos=2500.0 edad=35 historial_deuda=0 -> Aprobado


Respuesta: {'edad': 35, 'decision': 'Aprobado'}


## 2: Variables de entorno

Ahora vamos a proteger `/predict` con una API key: un texto secreto que quien consuma la API tiene que mandar para que le respondamos. Antes de escribir esa proteccion, hay que decidir donde vive esa clave.

La opcion incorrecta es escribirla directo en el codigo:

```python
API_KEY = "clave-super-secreta-2026"  # NO HACER ESO DENTRO DE main.py
```

El problema: ese archivo se sube a un repositorio de Git, queda dentro de la imagen de Docker, y cualquiera que tenga acceso al codigo o a la imagen tiene la clave. Si la clave se filtra o hay que cambiarla, hay que modificar el codigo y volver a construir la imagen.

La practica estandar es leer la clave desde una **variable de entorno**: un valor que vive afuera del codigo, en el sistema donde corre la aplicacion. En Python se lee con `os.environ`.

In [12]:
import os

# Sin la variable definida, no existe
print("Antes de definirla:", os.environ.get("API_KEY"))

# En una terminal real, esto se haria con: export API_KEY=clave-super-secreta-2026
# En Docker, se pasa con: docker run -e API_KEY=clave-super-secreta-2026 ...
# Aca en Colab no hay terminal, asi que la simulamos definiendola directamente:
os.environ["API_KEY"] = "clave-super-secreta-2026"

print("Despues de definirla:", os.environ.get("API_KEY"))

Despues de definirla: clave-super-secreta-2026


Es importante definir la variable de entorno **antes** de importar o recargar `main.py`, porque la vamos a leer una sola vez, cuando arranca la aplicacion (igual que se carga el modelo una sola vez al arrancar).

## 2.5: Un archivo .env para desarrollo local

Lo que acabamos de hacer (`os.environ["API_KEY"] = "..."`) funciona para probar en este notebook, pero no es como se trabaja en un proyecto real. Nadie quiere escribir esa linea a mano cada vez que arranca su API en la computadora. Para eso existe una practica muy comun: guardar las variables de entorno en un archivo `.env`, y cargarlas automaticamente al arrancar la aplicacion.

El flujo es el siguiente:

```
archivo .env
     |
load_dotenv()
     |
variable de entorno
     |
os.environ.get("API_KEY")
```

Vamos a necesitar dos archivos nuevos:

- `.env`: un archivo de texto simple, con una variable por linea (`NOMBRE=valor`). Este archivo **nunca se sube a Git**, porque ahi es donde vive el secreto real.
- `.gitignore`: le dice a Git que ignore ciertos archivos o carpetas. Le vamos a agregar `.env`, para que Git jamas lo registre, ni por accidente.

In [13]:
%%writefile .env
API_KEY=clave-super-secreta-2026

Writing .env


In [14]:
%%writefile .gitignore
.env
__pycache__/
*.pyc

Writing .gitignore


Ahora instalamos `python-dotenv`, la libreria que sabe leer un archivo `.env` ('hidden file') y cargar su contenido como variables de entorno.

In [15]:
!pip install -q python-dotenv

Y probamos que funcione. Primero limpiamos la variable que habiamos definido a mano en la Actividad 2, para asegurarnos de que el valor viene realmente del archivo `.env` y no de la celda anterior.

In [18]:
import os
from dotenv import load_dotenv

# Borramos la variable que definimos a mano, para probar limpio
os.environ.pop("API_KEY", None)
print("Antes de load_dotenv():", os.environ.get("API_KEY"))

load_dotenv()  # busca un archivo .env en la carpeta actual y carga sus variables

print("Despues de load_dotenv():", os.environ.get("API_KEY"))

Despues de load_dotenv(): clave-super-secreta-2026


In [19]:
# para ver las hidden files
!ls -la

total 36
drwxr-xr-x 1 root root 4096 Sep 30 15:25 .
drwxr-xr-x 1 root root 4096 Sep 30 15:03 ..
drwxr-xr-x 4 root root 4096 Sep 16 13:26 .config
-rw-r--r-- 1 root root   33 Sep 30 15:25 .env
-rw-r--r-- 1 root root   24 Sep 30 15:25 .gitignore
-rw-r--r-- 1 root root 1896 Sep 30 15:21 main.py
-rw-r--r-- 1 root root 1183 Sep 30 15:21 model.pkl
drwxr-xr-x 2 root root 4096 Sep 30 15:21 __pycache__
drwxr-xr-x 1 root root 4096 Sep 16 13:26 sample_data


`load_dotenv()` hizo el trabajo: leyo el archivo `.env` y dejo `API_KEY` disponible en `os.environ`, exactamente como si la hubieramos definido a mano. La diferencia es que ahora ese valor vive en un archivo separado, versionado (o mejor dicho, no versionado) aparte del codigo.

Si este fuera un proyecto real, `requirements.txt` tambien deberia incluir `python-dotenv`, y `main.py` empezaria con `from dotenv import load_dotenv` seguido de `load_dotenv()`, antes de leer cualquier variable.

## Actividad 3: Proteger /predict con una API key

FastAPI tiene un mecanismo llamado `Depends()` para las llamadas "dependencias": codigo que se ejecuta antes que la funcion del endpoint, y que puede bloquear la peticion antes de que llegue a la logica principal. Es la misma herramienta que se usaria para verificar un token de autenticacion en una API real.

Vamos a usar `APIKeyHeader` (que viene incluido en FastAPI) para leer un header llamado `X-API-Key`, y una funcion `verificar_api_key` que compara ese valor contra la variable de entorno. Si no coincide, respondemos con un error `401 Unauthorized`, sin ejecutar la prediccion.

In [22]:
%%writefile main.py
import logging
import os

import joblib
from fastapi import Depends, FastAPI, HTTPException, status  # Depends permite usar dependencias en los endpoints
from fastapi.security import APIKeyHeader  # Permite recibir la API key desde un header HTTP
from pydantic import BaseModel

# Configuración del sistema de logging
logger = logging.getLogger("scoring_api")           # Creamos un logger específico para nuestra API
logger.setLevel(logging.INFO)                       # Indicamos que queremos registrar mensajes INFO o más importantes
logger.propagate = False                            # Evitamos que el mensaje se duplique en el logger principal

if not logger.handlers:                             # Evitamos crear varios handlers si el módulo se recarga
    handler = logging.StreamHandler()               # Envía los logs a la consola
    handler.setLevel(logging.INFO)                  # El handler muestra mensajes INFO o más importantes

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"  # Formato: fecha/hora | nivel | mensaje
    )
    handler.setFormatter(formatter)                 # Aplicamos el formato al handler
    logger.addHandler(handler)                      # Conectamos el handler con nuestro logger



API_KEY = os.environ.get("API_KEY")  # Obtiene la API key desde una variable de entorno
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)  # Define el header donde el cliente enviará la API key


app = FastAPI(title="API de Scoring Crediticio")
modelo = joblib.load("model.pkl")

total_predicciones = 0


class ClienteInput(BaseModel):
    ingresos: float
    edad: int
    historial_deuda: int


def verificar_api_key(api_key: str = Depends(api_key_header)):  # Depends obtiene la API key del header antes de ejecutar la función
    if api_key != API_KEY:  # Compara la API key recibida con la API key correcta
        logger.warning("Intento de acceso con API key invalida o faltante")  # Registra un intento de acceso no autorizado
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="API key invalida o faltante"
        )
    return api_key  # Devuelve la API key si es válida


@app.get("/health")
def health():
    return {"status": "online"}


@app.post("/predict")
def predict(cliente: ClienteInput, api_key: str = Depends(verificar_api_key)): # Depends ejecuta la validación antes de permitir el acceso
    global total_predicciones

    datos = [[cliente.ingresos, cliente.edad, cliente.historial_deuda]]
    prediccion = modelo.predict(datos)[0]
    decision = "Aprobado" if prediccion == 1 else "Rechazado"

    total_predicciones += 1
    logger.info(
        "Prediccion #%s | ingresos=%s edad=%s historial_deuda=%s -> %s",
        total_predicciones, cliente.ingresos, cliente.edad, cliente.historial_deuda, decision
    )

    return {"edad": cliente.edad, "decision": decision}

Overwriting main.py


Notas del codigo:

- `APIKeyHeader(name="X-API-Key", auto_error=False)` define que header vamos a leer, y `auto_error=False` evita que FastAPI tire un error generico antes de que nosotros podamos decidir el mensaje.
- `/health` queda **sin proteccion a proposito**: es comun que un servicio de monitoreo (o el propio Docker, con `HEALTHCHECK`) necesite consultar la salud de la API sin tener que manejar credenciales.
- `/predict` ahora depende de `verificar_api_key`, ademas de depender de `ClienteInput`. FastAPI ejecuta ambas dependencias antes de correr el cuerpo de la funcion.

Probamos tres escenarios: sin header, con la clave incorrecta, y con la clave correcta.

In [23]:
importlib.reload(main)
client = TestClient(main.app)

cliente_ejemplo = {"ingresos": 2500.0, "edad": 35, "historial_deuda": 0}

print("--- Sin header ---")
r = client.post("/predict", json=cliente_ejemplo)
print(r.status_code, r.json())

print("--- Con clave incorrecta ---")
r = client.post("/predict", json=cliente_ejemplo, headers={"X-API-Key": "clave-equivocada"})
print(r.status_code, r.json())

print("--- Con clave correcta ---")
r = client.post("/predict", json=cliente_ejemplo, headers={"X-API-Key": "clave-super-secreta-2026"})
print(r.status_code, r.json())

print("--- /health sigue sin pedir clave ---")
print(client.get("/health").json())

2026-09-30 15:32:50,795 | WARNING | Intento de acceso con API key invalida o faltante
2026-09-30 15:32:50,798 | WARNING | Intento de acceso con API key invalida o faltante
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
2026-09-30 15:32:50,802 | INFO | Prediccion #1 | ingresos=2500.0 edad=35 historial_deuda=0 -> Aprobado


--- Sin header ---
401 {'detail': 'API key invalida o faltante'}
--- Con clave incorrecta ---
401 {'detail': 'API key invalida o faltante'}
--- Con clave correcta ---
200 {'edad': 35, 'decision': 'Aprobado'}
--- /health sigue sin pedir clave ---
{'status': 'online'}


## Como se ve esto en Docker

Los archivos que ya conoces (`requirements.txt`, `Dockerfile`) no cambian en su estructura. Lo que cambia es **como se le pasa la clave al contenedor**, y ahi hay una regla importante: la clave real nunca deberia escribirse dentro del `Dockerfile` ni del codigo, porque eso quedaria grabado dentro de la imagen (cualquiera que tenga la imagen podria extraerla).

Con el `.env` que armamos en la Actividad 2.5 hay que tener cuidado con algo puntual: si tu `Dockerfile` tiene una linea `COPY . .`, y no le decis lo contrario, **copiaria el archivo `.env` dentro de la imagen**, con el secreto adentro. Por eso el `.dockerignore` (que ya usamos en la Sesion 2 para excluir `.git` y `__pycache__`) tambien deberia excluir `.env`:

```
.env
.git
__pycache__/
```

De hecho, dentro del contenedor **no hace falta `load_dotenv()` en absoluto**: Docker ya te da una forma de inyectar variables de entorno al arrancar el contenedor (`-e` o `--env-file`), asi que `os.environ.get("API_KEY")` va a encontrar el valor igual, sin necesidad de leer ningun archivo `.env`. La libreria `python-dotenv` es una herramienta para el desarrollo local, no para dentro del contenedor.

Con esto queda un progreso de tres pasos, cada uno mas cercano a produccion:

```
.env + .gitignore          ->   Docker -e / --env-file   ->   gestor de secretos
(desarrollo local)              (contenedor)                   (produccion real)
```


La forma correcta de pasar la clave al momento de correr el contenedor, sin depender del `.env`, es:

```
docker run -p 8000:8000 -e API_KEY=clave-super-secreta-2026 ml-api-app
```

El flag `-e` define una variable de entorno solo para ese contenedor en ejecucion. Otra opcion, cuando hay varias variables, es un archivo `.env` (que nunca se sube a Git):

```
# archivo .env
API_KEY=clave-super-secreta-2026
```

```
docker run -p 8000:8000 --env-file .env ml-api-app
```


## Cierre

Con esto le diste a tu API dos piezas reales del "Nivel 6: Produccion" que quedaron pendientes en el cierre del laboratorio: un registro de lo que va pasando (logging) y una barrera basica de acceso (API key por variable de entorno). Es una version simple de lo que existe en produccion: ahi la clave se guardaria en un gestor de secretos (no en un archivo `.env` suelto), y los logs irian a un sistema centralizado en vez de a la consola. Pero el mecanismo de fondo, dependencias de FastAPI y variables de entorno leidas en tiempo de ejecucion, es exactamente el mismo.